# Data collection
## 1. Environment and folder setup

In [13]:
import os
# 1.1 Install requests
try:
    import requests
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "requests"])
    import requests

import json
from typing import Generator, Dict, Any, List
# 1.2 Project root directory
PROJECT_ROOT = r"C:\Users\b1795\Desktop\Laboratory2_SP"
# 1.3 Raw data directory
DATA_RAW_DIR = os.path.join(PROJECT_ROOT, "data_raw")
os.makedirs(DATA_RAW_DIR, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", DATA_RAW_DIR)

Project root: C:\Users\b1795\Desktop\Laboratory2_SP
Raw data folder: C:\Users\b1795\Desktop\Laboratory2_SP\data_raw


## 2. UniProt API helpers (pagination and JSON parsing)  

In [14]:
def uniprot_search_generator(first_url: str) -> Generator[Dict[str, Any], None, None]:
    url = first_url  # Retrieve all results via the UniProt REST API using pagination.
    while url is not None:
        print(f"Requesting: {url[:120]}:")  # Print the first 120 characters for inspection
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        results = data.get("results", [])
        for entry in results:
            yield entry
        # Parse the Link field in the HTTP header to find the next page cursor
        link_header = response.headers.get("Link")
        next_url = None
        if link_header:
            parts = [p.strip() for p in link_header.split(",")]
            for part in parts:
                if 'rel="next"' in part:
                    start = part.find("<") + 1
                    end = part.find(">")
                    next_url = part[start:end]
                    break
        url = next_url 
def get_sequence(entry: Dict[str, Any]) -> str:
    # Extract the amino acid sequence string from an entry.
    seq = entry.get("sequence", {}).get("value")
    if not seq:
        raise ValueError(f"No sequence found for entry {entry.get('primaryAccession')}")
    return seq
def get_protein_length(entry: Dict[str, Any]) -> int:
    # Get the protein length from an entry.
    length = entry.get("sequence", {}).get("length")
    if length is None:
        length = len(get_sequence(entry))
    return int(length)
def get_accession(entry: Dict[str, Any]) -> str:
    # Primary accession.
    return entry.get("primaryAccession")
def get_organism_name(entry: Dict[str, Any]) -> str:
    # Organism name (scientificName).
    return entry.get("organism", {}).get("scientificName", "Unknown organism")
def get_kingdom(entry: Dict[str, Any]) -> str:
    """
    Roughly classify eukaryotes into Metazoa / Fungi / Plants / Other.
    Perform a simple classification based on keywords in the taxonomy lineage.
    """
    lineage = entry.get("organism", {}).get("lineage", []) or []
    lineage = [x.lower() for x in lineage]
    if "metazoa" in lineage:
        return "Metazoa"
    if "fungi" in lineage:
        return "Fungi"
    # For plants, UniProt commonly uses 'viridiplantae'
    if "viridiplantae" in lineage or "streptophyta" in lineage:
        return "Plants"
    return "Other"

## 3. Positive dataset helpers (signal peptide features)  

In [15]:
def find_signal_peptide_feature(entry: Dict[str, Any]) -> Dict[str, Any]:
    for feat in entry.get("features", []):
        if feat.get("type", "").lower() == "signal":
            return feat
    return None
def get_sp_cleavage_site_if_valid(entry: Dict[str, Any]) -> int:
    feat = find_signal_peptide_feature(entry)
    if feat is None:
        return None
    loc = feat.get("location", {})
    start = loc.get("start", {}).get("value")
    end = loc.get("end", {}).get("value")
    description = feat.get("description", "")
    # Exclude Not cleaved or unknown-location cases
    if end is None:
        return None
    if isinstance(end, str) and end == "?":
        return None
    if isinstance(description, str) and "not cleaved" in description.lower():
        return None
    try:
        start = int(start)
        end = int(end)
    except Exception:
        return None
    sp_length = end - start + 1
    if sp_length < 14:  # Signal peptide length must be > 13
        return None
    return end

## 4. Negative dataset helpers

In [16]:
def has_tm_helix_in_first_90(entry: Dict[str, Any]) -> bool:
    for feat in entry.get("features", []):
        ftype = feat.get("type", "").lower()
        if "transmembrane" not in ftype:
            continue
        loc = feat.get("location", {})
        start = loc.get("start", {}).get("value")
        try:
            start = int(start)
        except Exception:
            continue
        if start <= 90:
            return True
    return False

## 5. Collect positive and negative datasets from UniProt  

In [17]:
import csv

# 5.1 Define two API URLs
POSITIVE_URL = (
    "https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28taxonomy_id%3A2759%29+AND+%28reviewed%3Atrue%29+AND+%28fragment%3Afalse%29+AND+%28existence%3A1%29+AND+%28length%3A%5B40+TO+*%5D%29+AND+%28ft_signal_exp%3A*%29%29&size=500"
)
NEGATIVE_URL = (
    "https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28taxonomy_id%3A2759%29+AND+%28reviewed%3Atrue%29+AND+%28fragment%3Afalse%29+AND+%28existence%3A1%29+AND+%28length%3A%5B40+TO+*%5D%29+AND+NOT+%28ft_signal%3A*%29+AND+%28%28cc_scl_term_exp%3ASL-0091%29+OR+%28cc_scl_term_exp%3ASL-0191%29+OR+%28cc_scl_term_exp%3ASL-0173%29+OR+%28cc_scl_term_exp%3ASL-0209%29+OR+%28cc_scl_term_exp%3ASL-0204%29+OR+%28cc_scl_term_exp%3ASL-0039%29%29%29&size=500"
)

POS_TSV = os.path.join(DATA_RAW_DIR, "positive.tsv")
POS_FASTA = os.path.join(DATA_RAW_DIR, "positive.fasta")
NEG_TSV = os.path.join(DATA_RAW_DIR, "negative.tsv")
NEG_FASTA = os.path.join(DATA_RAW_DIR, "negative.fasta")

def collect_datasets(force_download: bool = False) -> None:
    all_exist = all(os.path.exists(p) for p in [POS_TSV, POS_FASTA, NEG_TSV, NEG_FASTA])
    if all_exist and not force_download:
        print("All output files already exist. Set force_download=True to re-download.")
        return
    # 5.2 Positive dataset
    print("\nCollecting POSITIVE dataset:")
    with open(POS_TSV, "w", newline="", encoding="utf-8") as tsv_f, \
            open(POS_FASTA, "w", encoding="utf-8") as fasta_f:
        tsv_writer = csv.writer(tsv_f, delimiter="\t")
        # TSV header
        tsv_writer.writerow([
            "accession",
            "organism",
            "kingdom",
            "length",
            "sp_cleavage_site"
        ])
        count_total = 0
        count_used = 0
        for entry in uniprot_search_generator(POSITIVE_URL):
            count_total += 1
            acc = get_accession(entry)
            length = get_protein_length(entry)

            # Additional checks: SP cleavage site + SP length >= 14
            cleavage_end = get_sp_cleavage_site_if_valid(entry)
            if cleavage_end is None:
                # Skip positive entries that do not satisfy slide conditions
                continue
            organism = get_organism_name(entry)
            kingdom = get_kingdom(entry)
            seq = get_sequence(entry)
            # Write TSV
            tsv_writer.writerow([acc, organism, kingdom, length, cleavage_end])
            # Write FASTA
            fasta_f.write(f">{acc} {organism}\n")
            for i in range(0, len(seq), 60):
                fasta_f.write(seq[i:i+60] + "\n")
            count_used += 1
    print(f"Positive entries retrieved: {count_total}, kept after SP checks: {count_used}")
    print("Positive TSV:", POS_TSV)
    print("Positive FASTA:", POS_FASTA)

    # 5.3 Negative dataset
    print("\nCollecting NEGATIVE dataset:")
    with open(NEG_TSV, "w", newline="", encoding="utf-8") as tsv_f, \
            open(NEG_FASTA, "w", encoding="utf-8") as fasta_f:
        tsv_writer = csv.writer(tsv_f, delimiter="\t")
        # TSV header
        tsv_writer.writerow([
            "accession",
            "organism",
            "kingdom",
            "length",
            "tm_helix_start_leq_90"  # true / false
        ])
        count_total = 0
        count_used = 0
        for entry in uniprot_search_generator(NEGATIVE_URL):
            count_total += 1
            acc = get_accession(entry)
            length = get_protein_length(entry)
            organism = get_organism_name(entry)
            kingdom = get_kingdom(entry)
            seq = get_sequence(entry)
            # confirm again that no signal feature exists
            if find_signal_peptide_feature(entry) is not None:
                continue
            tm_flag = has_tm_helix_in_first_90(entry)
            # Write TSV
            tsv_writer.writerow([acc, organism, kingdom, length, str(tm_flag).lower()])
            # Write FASTA
            fasta_f.write(f">{acc} {organism}\n")
            for i in range(0, len(seq), 60):
                fasta_f.write(seq[i:i+60] + "\n")
            count_used += 1
    print(f"Negative entries retrieved: {count_total}, kept (no SP): {count_used}")
    print("Negative TSV:", NEG_TSV)
    print("Negative FASTA:", NEG_FASTA)

## 6. Run data collection

In [18]:
collect_datasets(force_download=True)


Requesting: https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28taxonomy_id%3A2759%29+AND+%28reviewed%3Atrue%29+AND+%2:
Requesting: https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28taxonomy_id%3A2759%29%20AND%20%28reviewed%3Atrue%29%20:
Requesting: https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28taxonomy_id%3A2759%29%20AND%20%28reviewed%3Atrue%29%20:
Requesting: https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28taxonomy_id%3A2759%29%20AND%20%28reviewed%3Atrue%29%20:
Requesting: https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28taxonomy_id%3A2759%29%20AND%20%28reviewed%3Atrue%29%20:
Requesting: https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28taxonomy_id%3A2759%29%20AND%20%28reviewed%3Atrue%29%20:
Positive entries retrieved: 2949, kept after SP checks: 2935
Positive TSV: C:\Users\b1795\Desktop\Laboratory2_SP\data_raw\positive.tsv
Positive FASTA: C:\Users\b1795\Desktop\Laboratory2_SP\data_r